In [1]:
!pip install -q "ultralytics==8.4.149"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 460.4 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from zipfile import ZipFile
import torch

from ultralytics import SAM
from ultralytics.models.sam import SAM2Predictor
import ultralytics

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

Ultralytics: 8.4.149
PyTorch: 2.11.0+cu128
Device: 0


**Paths**

In [5]:
DATASET_URL = (
    "https://github.com/ultralytics/"
    "assets/releases/download/v0.0.0/"
    "crack-seg.zip"
)

DATASETS_ROOT = Path("/content/datasets")
ARCHIVE_PATH = DATASETS_ROOT / "crack-seg.zip"
DATASET_ROOT = DATASETS_ROOT / "crack-seg"

In [6]:
DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "val", "test")
EXPECTED_COUNTS = {
    "train": 3717,
    "val": 200,
    "test": 112,
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

print("Dataset root:", DATASET_ROOT)

Dataset root: /content/datasets/crack-seg


In [7]:
dataset_ready = (DATASET_ROOT / "images" / "train").is_dir()

if not dataset_ready:
    if not ARCHIVE_PATH.is_file():
        print("Downloading Crack-Seg...")
        torch.hub.download_url_to_file(
            DATASET_URL, str(ARCHIVE_PATH), progress=True
        )
        
    print("Extracting dataset...")
    with ZipFile(ARCHIVE_PATH, "r") as zip_file:
        zip_file.extractall(DATASETS_ROOT)


if not DATASET_ROOT.is_dir():
    candidates = [
        path for path in DATASETS_ROOT.rglob("*")
        if path.is_dir() and (path / "images").is_dir() and "crack" in path.name.lower()
    ]

    if len(candidates) != 1:
        candidates = [
            path for path in DATASETS_ROOT.rglob("images")
            if path.is_dir()
        ]
        if len(candidates) == 1:
            candidates = [candidates[0].parent]

    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Could not locate Crack-Seg root in {DATASETS_ROOT}. "
            f"Found candidates: {candidates}"
        )

    DATASET_ROOT = candidates[0]

100%|██████████| 91.6M/91.6M [00:03<00:00, 24.4MB/s]


Extracting dataset...


In [8]:
for split in SPLITS:
    required_directories = [
        (DATASET_ROOT / "images" / split),
        (DATASET_ROOT / "labels" / split)
    ]
    
    for directory in (required_directories):
        if not directory.is_dir():
            raise FileExistsError(directory)

print("Dataset extracted:", DATASET_ROOT)

Dataset extracted: /content/datasets


In [9]:
VAL_IMAGES_DIR = DATASET_ROOT/ "images" / "val"
VAL_LABELS_DIR = DATASET_ROOT / "labels" / "val"

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_05/"
    "sam2_image"
)

FIGURE_DIR =  OUTPUT_ROOT / "figures"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


print("Validation images:", VAL_IMAGES_DIR)
print("Validation labels:", VAL_LABELS_DIR)
print("Outputs:", OUTPUT_ROOT)

Validation images: /content/datasets/images/val
Validation labels: /content/datasets/labels/val
Outputs: /content/drive/MyDrive/vision_unit_02_outputs/block_05/sam2_image


**Image index**

In [10]:
validation_image_index = {
    path.stem: path 
    for path in VAL_IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
}

print("Validation images:", len(validation_image_index))

Validation images: 200


**Polygon → separate instance masks**

In [11]:
def load_instance_masks(image_path, label_path):
    bgr_image = cv2.imread(str(image_path))
    
    if bgr_image is None:
        raise ValueError(f"Cannot read: {image_path}")

    rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    height, width = rgb_image.shape[:2]
    
    instance_masks = []
    
    label_text = label_path.read_text(encoding="utf-8").strip()
    if not label_text:
        return rgb_image, instance_masks
    
    for line_number, line in enumerate(label_text.splitlines(), start=1):
        values = line.split()
        
        if (len(values) < 7 or (len(values) - 1) % 2 != 0):
            raise ValueError(
                f"Invalid polygon: "
                f"{label_path.name}, "
                f"line {line_number}"
            )
        
        class_id = int(float(values[0]))
        if class_id != 0:
             raise ValueError(
                f"Unexpected class {class_id}"
            )
        
        normalized_points = np.asarray(values[1:], dtype=np.float32).reshape(-1, 2)
        
        if not np.all((normalized_points >= 0)& (normalized_points <= 1)):
            raise ValueError(
                f"Out-of-bounds polygon: "
                f"{label_path.name}"
            )
        
        pixel_points = np.empty_like(
            normalized_points, dtype=np.int32
        )
        
        pixel_points[:, 0] = np.clip(
            np.rint(normalized_points[:, 0] * width), 0, 
            width - 1
        ).astype(np.int32)
        
        pixel_points[:, 1] = np.clip(
            np.rint(normalized_points[:, 1] * height), 0, 
            height - 1
        ).astype(np.int32)
        
        mask = np.zeros((height, width), dtype=np.uint8)
        
        cv2.fillPoly(mask, [pixel_points], color=1)
        if mask.any():
            instance_masks.append(mask)
    
    return rgb_image, instance_masks

**Prompt and metric utilities**

In [12]:
def binary_iou(prediction, target):
    prediction = prediction > 0
    target = target > 0
    
    intersection = np.logical_and(prediction, target).sum()
    union = np.logical_or(prediction, target).sum()
    
    if union == 0:
        return np.nan
    
    return float(intersection / union)

In [32]:
def binary_dice(prediction, target):
    prediction = prediction > 0
    target = target > 0
    
    intersection = np.logical_and(prediction, target).sum()
    denominator = (prediction.sum() + target.sum())
    
    if denominator == 0:
        return np.nan

    return float(2 * intersection / denominator)

**Positive-point selection**

In [14]:
def find_positive_point(mask):
    distance = cv2.distanceTransform(
        mask.astype(np.uint8), cv2.DIST_L2, 5
    )
    
    y, x = np.unravel_index(
        np.argmax(distance), distance.shape
    )
    
    return int(x), int(y)

In [15]:
def mask_to_box(mask):
    ys, xs = np.where(mask > 0)
    
    if len(xs) == 0:
        raise ValueError("Empty instance mask.")
    
    return [
        int(xs.min()),
        int(ys.min()),
        int(xs.max()),
        int(ys.max()),
    ]

In [16]:
def find_negative_point(mask, padding=15,):
    height, width = mask.shape
    x1, y1, x2, y2 = mask_to_box(mask)

    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(width - 1, x2 + padding)
    y2 = min(height - 1, y2 + padding)

    background = (mask == 0).astype(np.uint8)

    distance = cv2.distanceTransform(
        background,cv2.DIST_L2, 5
    )

    valid_region = np.zeros_like(
        mask, dtype=bool
    )

    valid_region[y1:y2 + 1, x1:x2 + 1] = True
    valid_region &= (mask == 0)

    candidate_scores = np.where(
        valid_region,
        distance,
        -1,
    )

    y, x = np.unravel_index(
        np.argmax(candidate_scores),
        candidate_scores.shape,
    )

    return int(x), int(y)

**Build validation instance manifest**

In [17]:
instance_records = []

for label_path in sorted(VAL_LABELS_DIR.glob("*.txt")):
    image_path = validation_image_index.get(label_path.stem)

    if image_path is None:
        continue

    image, masks = load_instance_masks(
        image_path,
        label_path,
    )

    height, width = image.shape[:2]

    for instance_index, mask in enumerate(masks):
        foreground_pixels = int(mask.sum())

        instance_records.append(
            {
                "image_path": str(image_path),
                "label_path": str(label_path),
                "image_name": image_path.name,
                "instance_index": instance_index,
                "foreground_pixels": foreground_pixels,
                "foreground_ratio": foreground_pixels / (height * width),
            }
        )

In [18]:
instance_manifest = pd.DataFrame(instance_records)

print("Validation instances:", len(instance_manifest))
print(
    instance_manifest[
        [
            "foreground_pixels",
            "foreground_ratio",
        ]
    ].describe()
)

Validation instances: 249
       foreground_pixels  foreground_ratio
count         249.000000        249.000000
mean         2822.261044          0.016308
std          2054.272774          0.011871
min            21.000000          0.000121
25%          1310.000000          0.007570
50%          2562.000000          0.014804
75%          3779.000000          0.021837
max         11971.000000          0.069174


**Select nine representative instances**

In [19]:
lower_boundary = (
    instance_manifest["foreground_pixels"].quantile(0.33)
)

upper_boundary = (
    instance_manifest["foreground_pixels"].quantile(0.67)
)

In [20]:
def assign_size_group(pixel_count):
    if pixel_count <= lower_boundary:
        return "small"

    if pixel_count <= upper_boundary:
        return "medium"

    return "large"


instance_manifest["size_group"] = instance_manifest["foreground_pixels"].apply(
    assign_size_group
)

In [21]:
selected_groups = []

for size_group in ["small", "medium","large"]:
    group = (
        instance_manifest[instance_manifest["size_group"] == size_group]
        .sort_values("foreground_pixels")
        .reset_index(drop=True)
    )

    selected_indices = np.linspace(
        0, len(group) - 1,
        3, dtype=int
    )

    selected_groups.append(group.iloc[selected_indices])

In [22]:
selected_instances = pd.concat(selected_groups, ignore_index=True)

selection_path = OUTPUT_ROOT / "sam2_selected_instances.csv"
selected_instances.to_csv(selection_path, index=False)

selected_instances[
    [
        "image_name",
        "instance_index",
        "size_group",
        "foreground_pixels",
        "foreground_ratio",
    ]
]

,image_name,instance_index,size_group,foreground_pixels,foreground_ratio
0,2352.rf.13d894635b7d1c9765f6d1a2404156b7.jpg,1,small,21,0.000121
1,1971.rf.d029ad8edf2ff5e40a416a1399e143ae.jpg,3,small,894,0.005166
2,3228.rf.2bc533ca5e879aecb669ac56ea1c2373.jpg,0,small,1679,0.009702
3,2244.rf.e5270b51e6e7ea41debaf81f16273d7e.jpg,0,medium,1683,0.009725
4,1939.rf.3395d327f8a49d82491b18e04080fee7.jpg,1,medium,2562,0.014804
5,2040.rf.d10ea68f663f198606ec2a816fb7c35b.jpg,0,medium,3300,0.019069
6,3224.rf.ad00820ac18b68d77dc7e662e000df08.jpg,0,large,3308,0.019115
7,3066.rf.3e5fa3c6ece0d8f7337cdf619089fb27.jpg,0,large,4417,0.025524
8,3028.rf.323af1e3d52cd8dfcc7c01e46ce5947c.jpg,0,large,11971,0.069174


**Load SAM2.1 Small**

In [23]:
sam_model = SAM("sam2.1_s.pt")

sam_model.info()

Model summary: 364 layers, 46,060,354 parameters, 46,060,354 gradients


(364, 46060354, 46060354, 0.0)

In [ ]:
sam_predictor = SAM2Predictor(
    overrides={
        "model": "sam2.1_s.pt",
        "task": "segment",
        "mode": "predict",
        "device": DEVICE,
        "imgsz": 1024,
        "conf": 0.0,
        "verbose": False,
        "save": False,
    }
)

sam_predictor.setup_model(
    model=sam_model.model,
    verbose=False
)

**SAM result extraction**

In [24]:
def extract_sam_masks(result, target_shape):
    
    if result.masks is None or len(result.masks.data) == 0:
        return (
            np.zeros(
                (0, target_shape[0], target_shape[1]),
                dtype=np.uint8,
            ),
            np.zeros(0, dtype=np.float32),
        )

    masks = result.masks.data.detach().cpu().numpy()
    masks = (masks > 0.5).astype(np.uint8)

    target_height, target_width = target_shape
    resized_masks = []

    for mask in masks:
        if mask.shape != target_shape:
            mask = cv2.resize(
                mask,
                (target_width, target_height),
                interpolation=cv2.INTER_NEAREST,
            )
        resized_masks.append(mask)

    resized_masks = np.stack(resized_masks)

    if result.boxes is not None and result.boxes.conf is not None:
        scores = (
            result.boxes.conf.detach().cpu().numpy().astype(np.float32)
        )
    else:
        scores = np.zeros(
            len(resized_masks),
            dtype=np.float32,
        )

    return resized_masks, scores

**Prompt inference function**

In [28]:
def run_sam_prompt(image_path, image_shape, prompt_type, positive_point=None, negative_point=None, bounding_box=None):
    arguments = {
        "source": str(image_path)
    }
    
    if prompt_type == "point":
        arguments.update({
            "points" : [
                [list(positive_point)]
            ],
            "labels" : [[1]],
            "multimask_output" : True
        })
    elif prompt_type == "positive_negative":
        arguments.update({
            "points" : [
                [list(positive_point), list(negative_point)]
            ],
            "labels" : [[1, 0]],
            "multimask_output" : False
        })
    elif prompt_type == "box":
        arguments.update({
            "bboxes" : list(bounding_box),
            "multimask_output" : False
        })
    else:
        raise ValueError(f"Unknown prompt: {prompt_type}")
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start_time = time.perf_counter()
    
    results = sam_predictor(**arguments)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    latency_ms = (
        time.perf_counter() - start_time
    ) * 1000
    
    masks, scores = extract_sam_masks(results[0], image_shape)
    
    if len(masks) == 0:
        return {
            "masks": masks,
            "scores": scores,
            "best_index": None,
            "best_mask": np.zeros(
                image_shape,
                dtype=np.uint8,
            ),
            "best_score": np.nan,
            "latency_ms": latency_ms
        }
    
    best_index = int(np.argmax(scores))
    return {
        "masks": masks,
        "scores": scores,
        "best_index": best_index,
        "best_mask": masks[best_index],
        "best_score": float(scores[best_index]),
        "latency_ms": latency_ms
    }

**Warm-up**

In [29]:
warmup_record = selected_instances.iloc[0]

warmup_image, warmup_masks = load_instance_masks(
    Path(warmup_record["image_path"]),
    Path(warmup_record["label_path"]),
)

warmup_mask = warmup_masks[int(warmup_record["instance_index"])]
warmup_point = find_positive_point(warmup_mask)

_ = run_sam_prompt(
    image_path=warmup_record["image_path"],
    image_shape=warmup_mask.shape,
    prompt_type="point",
    positive_point=warmup_point,
)

print("Candidate masks:", len(_["masks"]))
print("Predicted quality scores:", _["scores"])
print("Selected mask index:", _["best_index"])

Candidate masks: 3
Predicted quality scores: [    0.16373     0.51346    0.010725]
Selected mask index: 1


**Evaluate three prompt strategies**

In [33]:
evaluation_rows = []
prediction_cache = {}

for _, record in selected_instances.iterrows():
    image_path = Path(record["image_path"])
    label_path = Path(record["label_path"])

    image, instance_masks = load_instance_masks(image_path, label_path)

    instance_index = int(record["instance_index"])
    ground_truth = instance_masks[instance_index]

    positive_point = find_positive_point(ground_truth)
    negative_point = find_negative_point(ground_truth)
    bounding_box = mask_to_box(ground_truth)

    output_row = {
        "image_name": record["image_name"],
        "instance_index": instance_index,
        "size_group": record["size_group"],
        "foreground_pixels": int(record["foreground_pixels"]),
        "positive_x": positive_point[0],
        "positive_y": positive_point[1],
        "negative_x": negative_point[0],
        "negative_y": negative_point[1],
    }

    prompt_settings = {
        "point": {
            "positive_point": positive_point,
        },
        "positive_negative": {
            "positive_point": positive_point,
            "negative_point": negative_point,
        },
        "box": {
            "bounding_box": bounding_box,
        },
    }
    
    for prompt_name, settings in prompt_settings.items():
        prediction = run_sam_prompt(
            image_path=image_path,
            image_shape=ground_truth.shape,
            prompt_type=prompt_name,
            **settings
        )
        
        output_row[f"{prompt_name}_iou"] = binary_iou(
            prediction["best_mask"],
            ground_truth
        )
        output_row[f"{prompt_name}_dice"] = binary_dice(
            prediction["best_mask"],
            ground_truth
        )
        
        output_row[f"{prompt_name}_score"] = prediction["best_score"]
        
        output_row[f"{prompt_name}_latency_ms"] = prediction["latency_ms"]
        
        output_row[f"{prompt_name}_candidates"] = len(prediction["masks"])
        
        cache_key = (
            record["image_name"],
            instance_index,
            prompt_name,
        )
        
        prediction_cache[cache_key] = prediction

    evaluation_rows.append(output_row)

In [34]:
sam2_metrics = pd.DataFrame(evaluation_rows)

metrics_path = OUTPUT_ROOT / "sam2_prompt_metrics.csv"

sam2_metrics.to_csv(metrics_path, index=False)
sam2_metrics

,image_name,instance_index,size_group,foreground_pixels,positive_x,positive_y,negative_x,negative_y,point_iou,point_dice,...,positive_negative_iou,positive_negative_dice,positive_negative_score,positive_negative_latency_ms,positive_negative_candidates,box_iou,box_dice,box_score,box_latency_ms,box_candidates
0,2352.rf.13d894635b7d1c9765f6d1a2404156b7.jpg,1,small,21,256,200,237,182,0.004941,0.009833,...,0.003715,0.007402,0.577748,231.881253,1,0.341463,0.509091,0.707484,169.052184,1
1,1971.rf.d029ad8edf2ff5e40a416a1399e143ae.jpg,3,small,894,354,122,376,210,0.346457,0.514620,...,0.392727,0.563969,0.567068,166.048052,1,0.534632,0.696756,0.749467,166.305409,1
2,3228.rf.2bc533ca5e879aecb669ac56ea1c2373.jpg,0,small,1679,164,28,107,9,0.696685,0.821231,...,0.689130,0.815959,0.812412,164.889492,1,0.674688,0.805748,0.839299,164.442950,1
3,2244.rf.e5270b51e6e7ea41debaf81f16273d7e.jpg,0,medium,1683,194,53,170,415,0.511650,0.676942,...,0.506032,0.672007,0.761516,164.242171,1,0.474667,0.643762,0.821926,167.507474,1
4,1939.rf.3395d327f8a49d82491b18e04080fee7.jpg,1,medium,2562,319,239,205,214,0.266064,0.420301,...,0.222567,0.364098,0.502436,164.152011,1,0.500364,0.666990,0.539825,163.771755,1
5,2040.rf.d10ea68f663f198606ec2a816fb7c35b.jpg,0,medium,3300,397,202,206,182,0.605107,0.753977,...,0.645815,0.784796,0.757776,164.545873,1,0.584203,0.737535,0.747456,164.744613,1
6,3224.rf.ad00820ac18b68d77dc7e662e000df08.jpg,0,large,3308,101,51,29,415,0.069831,0.130545,...,0.072854,0.135813,0.557534,166.181762,1,0.609591,0.757448,0.690809,166.702583,1
7,3066.rf.3e5fa3c6ece0d8f7337cdf619089fb27.jpg,0,large,4417,196,387,174,16,0.528957,0.691919,...,0.535259,0.697288,0.645087,163.781356,1,0.557099,0.715560,0.781948,168.705486,1
8,3028.rf.323af1e3d52cd8dfcc7c01e46ce5947c.jpg,0,large,11971,124,260,42,171,0.682540,0.811321,...,0.689214,0.816017,0.924400,165.077929,1,0.677856,0.808003,0.953869,166.863082,1


**Prompt comparison summary**

In [35]:
summary_rows = []

for prompt_name in [
    "point",
    "positive_negative",
    "box",
]:
    summary_rows.append(
        {
            "prompt": prompt_name,
            "mean_iou": sam2_metrics[f"{prompt_name}_iou"].mean(),
            "median_iou": sam2_metrics[f"{prompt_name}_iou"].median(),
            "mean_dice": sam2_metrics[f"{prompt_name}_dice"].mean(),
            "mean_predicted_quality": sam2_metrics[
                f"{prompt_name}_score"
            ].mean(),
            "mean_latency_ms": sam2_metrics[
                f"{prompt_name}_latency_ms"
            ].mean(),
        }
    )

prompt_summary = pd.DataFrame(summary_rows).sort_values(
    "mean_iou", ascending=False
)

In [37]:
summary_path = OUTPUT_ROOT / "sam2_prompt_summary.csv"

prompt_summary.to_csv(summary_path, index=False)
prompt_summary

,prompt,mean_iou,median_iou,mean_dice,mean_predicted_quality,mean_latency_ms
2,box,0.550507,0.557099,0.704544,0.759120,166.45506
1,positive_negative,0.417479,0.506032,0.539705,0.678442,172.31110
0,point,0.412470,0.511650,0.536743,0.637230,173.77794


**Performance by mask size**

In [38]:
size_summary_rows = []

for size_group in [
    "small",
    "medium",
    "large",
]:
    subset = sam2_metrics[sam2_metrics["size_group"] == size_group]
    size_summary_rows.append(
        {
            "size_group": size_group,
            "instances": len(subset),
            "point_mean_iou": subset["point_iou"].mean(),
            "positive_negative_mean_iou": subset["positive_negative_iou"].mean(),
            "box_mean_iou": subset["box_iou"].mean(),
        }
    )

size_summary = pd.DataFrame(size_summary_rows)
size_summary

,size_group,instances,point_mean_iou,positive_negative_mean_iou,box_mean_iou
0,small,3,0.349361,0.361858,0.516928
1,medium,3,0.460940,0.458138,0.519745
2,large,3,0.427109,0.432442,0.614849
